# 3. Model Architecture

Build a Siamese Network for Facial Verification based on the Nicholas Renotte tutorial.

**Architecture:**
- Input: Two 100x100x3 images
- Embedding Network: Conv2D + MaxPooling blocks → 4096 Dense
- L1 Distance Layer: Calculates the difference between embeddings
- Classifier: Dense(1, sigmoid) → Similarity Score [0, 1]

## 3.1 Import & Setup

In [1]:
import sys
from pathlib import Path
import tensorflow as tf
from src.models import L1Dist
import keras
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Layer, Input, Conv2D, MaxPooling2D, Dense, Flatten

# Project
sys.path.insert(0, str(Path.cwd().parent))
import config
from src import models
from src import utils



print("✓ Imports successful")
print(f"TensorFlow Version: {tf.__version__}")

✓ Imports successful
TensorFlow Version: 2.16.2


## 3.2 Creating Embedding Network

In [2]:
print("\nCreating Embedding Network...\n")

embedding = models.create_embedding_network()
print(f"✓ Embedding Network created")

# Summary
models.print_model_summary(embedding)


Creating Embedding Network...

✓ Embedding Network created

MODEL SUMMARY: embedding_network


Model: "embedding_network"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_image (InputLayer)        │ (None, 100, 100, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 91, 91, 64)     │        19,264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ maxpool_1 (MaxPooling2D)        │ (None, 46, 46, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 40, 40, 128)    │       401,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ maxpool_2 (MaxPooling2D)        │ (None, 20, 20, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 17, 17, 128)    │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ maxpool_3 (MaxPooling2D)        │ (None, 9, 9, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 6, 6, 256)      │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 9216)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_dropout (Dropout)     │ (None, 9216)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_embedding (Dense)         │ (None, 4096)           │    37,752,832 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 38,960,448 (148.62 MB)

 Trainable params: 38,960,448 (148.62 MB)

 Non-trainable params: 0 (0.00 B)

## 3.3 Model Statistics

In [3]:
# Model Info
info = models.get_model_info(embedding)

print("\n" + "="*80)
print("EMBEDDING NETWORK DETAILS")
print("="*80)

for key, value in info.items():
    if isinstance(value, (int, float)):
        if 'params' in key:
            print(f"{key}: {value:,}")
        else:
            print(f"{key}: {value}")
    else:
        print(f"{key}: {value}")

print("\n" + "="*80)


EMBEDDING NETWORK DETAILS
name: embedding_network
layers: 11
total_params: 38,960,448
trainable_params: 38960448
non_trainable_params: 0
input_shape: (None, 100, 100, 3)
output_shape: (None, 4096)



## 3.4 Test Forward Pass

In [4]:
# Test Forward Pass
print("\nTesting Forward Pass with Dummy Input...\n")

dummy_input = tf.random.normal((1, config.IMG_SIZE, config.IMG_SIZE, config.IMG_CHANNELS))
output = embedding(dummy_input, training=False)

print(f"Input Shape: {dummy_input.shape}")
print(f"Output Shape: {output.shape}")
print(f"Output Value (sample): {output[0, :5].numpy()}")
print(f"Output Range: [{output.numpy().min():.6f}, {output.numpy().max():.6f}]")
print("\n✓ Forward Pass successful")


Testing Forward Pass with Dummy Input...

Input Shape: (1, 100, 100, 3)
Output Shape: (1, 4096)
Output Value (sample): [0.48022035 0.48188734 0.47428957 0.4509436  0.52026063]
Output Range: [0.358120, 0.632626]

✓ Forward Pass successful


## 3.5 Create Siamese Network

In [5]:
print("\nCreate Siamese Network...\n")

siamese_model = models.create_siamese_network()
print(f"✓ Siamese Network create")

# Summary
models.print_model_summary(siamese_model)


Create Siamese Network...

✓ Siamese Network create

MODEL SUMMARY: siamese_network


Model: "siamese_network"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ anchor_image        │ (None, 100, 100,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ verification_image  │ (None, 100, 100,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_network   │ (None, 4096)      │ 38,960,448 │ anchor_image[0][… │
│ (Functional)        │                   │            │ verification_ima… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ l1_distance         │ (None, 4096)      │          0 │ embedding_networ… │
│ (L1Dist)            │                   │            │ embedding_networ… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ classifier_dropout  │ (None, 4096)      │          0 │ l1_distance[0][0] │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ similarity_score    │ (None, 1)         │      4,097 │ classifier_dropo… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 38,964,545 (148.64 MB)

 Trainable params: 38,964,545 (148.64 MB)

 Non-trainable params: 0 (0.00 B)

## 3.6 Siamese Model Statistics

In [6]:
# Model Info
info = models.get_model_info(siamese_model)

print("\n" + "="*80)
print("SIAMESE NETWORK DETAILS")
print("="*80)

print(f"\nModel Architecture:")
for key, value in info.items():
    if isinstance(value, (int, float)):
        if 'params' in key:
            print(f"  {key}: {value:,}")
        else:
            print(f"  {key}: {value}")
    else:
        print(f"  {key}: {value}")

print(f"\nInput Specification:")
for i, inp in enumerate(siamese_model.inputs):
    print(f"  Input {i+1}: {inp.shape}")

print(f"\nOutput Specification:")
print(f"  Output: {siamese_model.output.shape}")
print(f"  Output Range: [0, 1] (sigmoid)")
print(f"  Meaning: 0=different, 1=same person")

print("\n" + "="*80)


SIAMESE NETWORK DETAILS

Model Architecture:
  name: siamese_network
  layers: 6
  total_params: 38,964,545
  trainable_params: 38964545
  non_trainable_params: 0
  input_shape: [(None, 100, 100, 3), (None, 100, 100, 3)]
  output_shape: (None, 1)

Input Specification:
  Input 1: (None, 100, 100, 3)
  Input 2: (None, 100, 100, 3)

Output Specification:
  Output: (None, 1)
  Output Range: [0, 1] (sigmoid)
  Meaning: 0=different, 1=same person



## 3.7 Test Siamese Forward Pass

In [7]:
print("\nTesting Siamese Forward Pass...\n")

# Dummy Inputs
anchor_input = tf.random.normal((2, config.IMG_SIZE, config.IMG_SIZE, config.IMG_CHANNELS))
verification_input = tf.random.normal((2, config.IMG_SIZE, config.IMG_SIZE, config.IMG_CHANNELS))

# Forward Pass
similarity_scores = siamese_model([anchor_input, verification_input], training=False)

print(f"Anchor Input: {anchor_input.shape}")
print(f"Verification Input: {verification_input.shape}")
print(f"Similarity Scores: {similarity_scores.shape}")
print(f"Values: {similarity_scores.numpy().flatten()}")
print(f"\nInterpretation:")
for i, score in enumerate(similarity_scores.numpy().flatten()):
    is_same = "SAME" if score >= config.VERIFICATION_THRESHOLD else "DIFFERENT"
    print(f"  Sample {i+1}: {score:.4f} → {is_same} (threshold={config.VERIFICATION_THRESHOLD})")

print("\n✓ Siamese Forward Pass successful")


Testing Siamese Forward Pass...

Anchor Input: (2, 100, 100, 3)
Verification Input: (2, 100, 100, 3)
Similarity Scores: (2, 1)
Values: [0.49655196 0.49593443]

Interpretation:
  Sample 1: 0.4966 → SAME (threshold=0.25)
  Sample 2: 0.4959 → SAME (threshold=0.25)

✓ Siamese Forward Pass successful


## 3.8 Model Summary & Next Steps

In [8]:
print("\n" + "="*80)
print("MODEL ARCHITECTURE SUMMARY")
print("="*80)

print(f"\n✓ Embedding Network:")
print(f"  - Input: {config.IMG_SIZE}x{config.IMG_SIZE}x{config.IMG_CHANNELS}")
print(f"  - 4 Conv2D blocks with MaxPooling")
print(f"  - Flatten → Dense(4096, sigmoid)")
print(f"  - Parameters: {info['total_params']:,}")

print(f"\n✓ Siamese Network:")
print(f"  - 2 Input Branches (Anchor + Verification)")
print(f"  - Shared Embedding Network (Weight Sharing)")
print(f"  - L1 Distance Layer (Manhattan Distance)")
print(f"  - Classification: Dense(1, sigmoid)")
print(f"  - Output: Similarity Score [0, 1]")

print(f"\n✓ Ready for Training!")
print(f"\nNext Step:")
print(f"  → Notebook 04_training.ipynb")

print("\n" + "="*80)
print("✓ Notebook 03 completed!")
print("="*80)


MODEL ARCHITECTURE SUMMARY

✓ Embedding Network:
  - Input: 100x100x3
  - 4 Conv2D blocks with MaxPooling
  - Flatten → Dense(4096, sigmoid)
  - Parameters: 38,964,545

✓ Siamese Network:
  - 2 Input Branches (Anchor + Verification)
  - Shared Embedding Network (Weight Sharing)
  - L1 Distance Layer (Manhattan Distance)
  - Classification: Dense(1, sigmoid)
  - Output: Similarity Score [0, 1]

✓ Ready for Training!

Next Step:
  → Notebook 04_training.ipynb

✓ Notebook 03 completed!
